# Register an Azure ML Model

Register a small transparent JSON model as an immutable custom-model asset and retrieve its metadata.

**Source:** Adapted from [Azure/azureml-examples model.ipynb](https://github.com/Azure/azureml-examples/blob/70f0ddd0cf92fb54af4031db0bb1dbd4a8443543/sdk/python/assets/model/model.ipynb), MIT License.

In [ ]:
from pathlib import Path
from azure.ai.ml import MLClient
from azure.identity import AzureCliCredential, ManagedIdentityCredential
from dotenv import dotenv_values
import json
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Model

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = Path(notebook_file).resolve().parent if notebook_file else Path.cwd().resolve()
for parent in (search_start, *search_start.parents):
    candidate = parent if parent.name == "workspace_readiness" else parent / "notebooks/workspace_readiness"
    if (candidate / ".env.example").is_file():
        SUITE_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run from this repository or notebooks/workspace_readiness.")

SAMPLE_ROOT = SUITE_ROOT.parents[1] / "workshop"

env_path = SUITE_ROOT / ".env"
if not env_path.is_file():
    raise FileNotFoundError(f"Configure {env_path} with your four Azure resource values.")
env = dotenv_values(env_path, encoding="utf-8-sig", interpolate=False)
ENV_KEYS = {
    "AZURE_SUBSCRIPTION_ID", "AZURE_RESOURCE_GROUP",
    "AZUREML_WORKSPACE_NAME", "AZUREML_REGISTRY_NAME",
    "AZUREML_COMPUTE_NAME", "AZUREML_COMPUTE_IDENTITY_CLIENT_ID",
    "USE_MANAGED_IDENTITY",
}
SUBSCRIPTION_ID = (env.get("AZURE_SUBSCRIPTION_ID") or "").strip()
RESOURCE_GROUP = (env.get("AZURE_RESOURCE_GROUP") or "").strip()
WORKSPACE_NAME = (env.get("AZUREML_WORKSPACE_NAME") or "").strip()
REGISTRY_NAME = (env.get("AZUREML_REGISTRY_NAME") or "").strip()
COMPUTE_NAME = (env.get("AZUREML_COMPUTE_NAME") or "").strip() or None
COMPUTE_IDENTITY_CLIENT_ID = (env.get("AZUREML_COMPUTE_IDENTITY_CLIENT_ID") or "").strip() or None

if set(env) - ENV_KEYS:
    raise ValueError(f"Unexpected .env keys: {', '.join(sorted(set(env) - ENV_KEYS))}.")
if not all((SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME)):
    raise ValueError("Fill in the four required workspace and registry values in .env.")
if any(value.startswith(("<", "${")) for value in (SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, REGISTRY_NAME, COMPUTE_NAME or "", COMPUTE_IDENTITY_CLIENT_ID or "")):
    raise ValueError("Replace the .env placeholders with your Azure resource values.")

use_managed_identity = (env.get("USE_MANAGED_IDENTITY", "true") or "").strip().lower()
if use_managed_identity not in {"true", "false"}:
    raise ValueError("USE_MANAGED_IDENTITY must be true or false in .env.")
credential = ManagedIdentityCredential(process_timeout=60) if use_managed_identity == "true" else AzureCliCredential(process_timeout=60)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
print(f"Workspace: {WORKSPACE_NAME}; registry: {REGISTRY_NAME}")

ASSET_NAMES = {
    "data": "readiness-taxi-data",
    "environment": "readiness-taxi-environment",
    "model": "readiness-taxi-model",
}



MODEL_NAME = ASSET_NAMES["model"]


In [ ]:
model_dir = SAMPLE_ROOT / "data/models/demo"
model_document = json.loads((model_dir / "model.json").read_text(encoding="utf-8"))

model_definition = Model(
    name=MODEL_NAME,
    type=AssetTypes.CUSTOM_MODEL,
    path=str(model_dir),
    description="Transparent two-feature taxi baseline used by the Azure ML workshop",
    tags={
        "validation": "workspace-readiness",
        "format": "json-linear-model",
        "features": ",".join(model_document["features"]),
    },
)

registered_model = ml_client.models.create_or_update(model_definition)
verified_model = ml_client.models.get(
    MODEL_NAME,
    version=registered_model.version,
)
assert verified_model.name == MODEL_NAME
assert verified_model.version == registered_model.version
print(f"Registered model: {verified_model.name}:{verified_model.version}")


## Expected Result

The transparent JSON model is registered as an immutable custom-model asset with workshop provenance tags.

Next: `05_submit_command_job.ipynb`.